# Marking Oracles Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the Marking Oracles kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import Qubits


## Problem 1. K-th bit

Since the effect of this oracle depends only on the value of the $k$-th qubit, we can ignore the rest of the qubits and focus on just `x[k]`. We need to flip the state of the target qubit if the input qubit `x[k]` is in the $\ket{1}$ state and leave it unchanged otherwise - this is exactly the effect of the controlled X gate with the qubit `x[k]` used as control and the qubit `y` used as target.

In [ ]:
def oracle_kth_bit(x: Qubits, y: Qubits, k: int) -> None:
    y.x(cond=x[k])

## Problem 2. Parity function

We can write $f(x)$ as follows:

$$f(x) = x_0 \oplus x_1 \oplus ... \oplus x_{N-1}$$

Let's substitute this expression in the expression for the oracle effect on the quantum state:

$$U_f \ket{x} \ket{y} = \ket{x} \ket{y \oplus f(x)} = \ket{x} \ket{y \oplus x_0 \oplus x_1 \oplus ... \oplus x_{N-1}}$$

Now, we can represent the final state of the target qubit as a result of a series of $N$ marking oracles applied sequentially, marking oracle $j$ flipping its state if the state of the input qubit $j$ $\ket{x_j}$ is $\ket{1}$.
As we saw in the previous problem, each of these marking oracles can be implemented as a single controlled $X$ gate.

In [ ]:
def oracle_parity(x: Qubits, y: Qubits) -> None:
    for xi in x:
        y.x(cond=xi)

## Problem 3. Product function

This problem is similar to the previous one, but this time the input qubit `x[j]` affects the state of the target qubit only if the classical input `r[i]` is set to `true`. 
We can use a similar approach to the solution as well: iterate through all input qubits, check whether the corresponding input bit `r[i]` is `true`, and if it is, apply a controlled $X$ gate with qubit `x[j]` as the control and the qubit `y` as the target.

In [ ]:
def oracle_product(x: Qubits, y: Qubits, r: list[bool]) -> None:
    for ind in range(x.num_qubits):
        if r[ind]:
            y.x(cond=x[ind])

## Problem 4. Product function with negation

Again, this problem is similar to the previous one. This time, each input qubit `x[j]` always affects the state of the target qubit, and the matching classical input `r[i]` specifies the way how it does that. 
If the bit `r[i]` is `True`, the state of the target qubit is flipped if the input qubit `x[j]` is in the $\ket{1}$ state;
otherwise, it is flipped if `x[j]` is in the $\ket{0}$ state. (You can check that this is exactly what the formula 
$\left(r_i x_i \oplus (1 - r_i) (1 - x_i) \right)$ evaluates to!)

This means that we need to modify our solution to the previous problem to add a second clause to the if statement, to handle the case when `r[i]` is `False`. The gate we need to apply in this scenario is controlled-on-zero $X$. In Workbench, we can implement it using the same `x` method with `cond=~x[ind]`.

Finally, we can merge these two code paths: in both cases, we need to call the method `y.x`, the only difference is the control condition we pass to it as the `cond` argument. We can write this using Python conditional expression: `x[ind] if r[ind] else ~x[ind]`.

In [ ]:
def oracle_product_negation(x: Qubits, y: Qubits, r: list[bool]) -> None:
    for ind in range(x.num_qubits):
        y.x(cond=x[ind] if r[ind] else ~x[ind])
        # # Explicit if-else construct
        # if r[ind]:
        #     y.x(cond=x[ind])
        # else:
        #     y.x(cond=~x[ind])

## Problem 5. Is bit string a palindrome?

For a basis state $\ket{x_0 ... x_{N-1}}$ to be a palindrome, the state of the first qubit $x_0$ has to be the same as that of the last qubit $x_{N-1}$, the state of the second qubit $x_1$ - the same as that of the second-to-last qubit $x_{N-2}$, and so on.
In other words, we need to compute XORs of pairs of qubits, and if each of these XORs is $0$, the basis state is a palindrome.

Recall that to compute XOR of the states of two qubits in-place, you can use a controlled $X$ gate with one of them as control and the other as target. After this, the state of the target qubit will be exactly the XOR:

$$CX \ket{a}\ket{b} = \ket{a}\ket{a \oplus b}$$

We can use this to calculate the XORs of $x_{N-1}$ and $x_0$ (stored in qubit `x[0]`), $x_{N-2}$ and $x_1$ (stored in qubit `x[1]`), and so on. 
To do this, we can run a `for` loop from $0$ to $\lfloor \tfrac{N}{2} \rfloor$ and apply individual controlled $X$ gates:

```python
    for ind in range(N // 2):
        x[ind].x(cond=x[N - ind - 1])
```

Workbench offers an alternative syntax for doing a series of single-controlled gates on two registers, "zipping" together matching qubits of the registers and using the qubits of one register as controls and of the other register - as targets. You can use that syntax to "zip" the target register `x[:N // 2]` and the control register `x[N - 1:N - N // 2 - 1:-1]` (remember that you need to revert the order of qubits in the second register):

```python
    x[:N // 2].x(cond_zip=x[N - 1:N - N // 2 - 1:-1])
```

Finally, to check that all XORs are $0$, we can apply a controlled-on-zero $X$ gate with the qubits we used to calculate XORs (stored in `x[:N // 2]`) as controls and the target qubit as the target. As we've seen in the previous problem, you can express the condition "the control register `reg` must be in $\ket{0}$ state" as `cond=~reg` or `cond=reg == 0`.

Remember to uncompute the changes you did to the qubits of the input register. You can do this easily by just repeating the sequence of gates you used to compute the XORs, since all individual gates in that computation act on different qubits.

In [ ]:
def oracle_palindrome(x: Qubits, y: Qubits) -> None:
    N = x.num_qubits
    K = N // 2
    for ind in range(K):
        x[ind].x(cond=x[N - ind - 1])
    # # Or, more concisely, use cond_zip argument to apply multiple controlled-X gates at once
    # x[:K].x(cond_zip=x[N - 1:N - K - 1:-1])
    y.x(cond=x[:K] == 0)
    for ind in range(K):
        x[ind].x(cond=x[N - ind - 1])

## Problem 6. Is bit string periodic with period P?

This problem is similar to the previous one. To check whether the bit string is a palindrome, you had to compare the states of the qubits in symmetrical positions. This time, you have to compare the states of the qubits at a distance $P$ from each other.
For the basis state $\ket{x_0 ... x_{N-1}}$ to be periodic with period $P$, you need to check that the pairs of qubits $x_0$ and $x_P$, $x_1$ and $x_{P + 1}$, and so on are the same. 

We can use a similar approach to the solution as well: do the comparisons in-place using controlled $X$ gates. 

If you iterate through the pairs in order from left to right, starting with the pair $x_0$ and $x_P$, you'll need to make sure to store the XORs in the *left* qubit of the pair. When comparing the states of qubits $x_j$ and $x_{j + P}$, the right qubit $x_{j + P}$ might be involved in another comparison later, if the position $j + 2P$ is within the bit string, so you shouldn't modify its state in the earlier comparison. However, the qubit $x_j$ had already been compared with the qubit $x_{j-P}$, so it's safe to modify its state now.

Same as in the previous problem, we then check that all XORs are $0$ using controlled-on-zero $X$ gate (with the first $N-P$ qubits as controls). 

Finally, we uncompute any changes we did to the input register. Notice that this time the order of controlled $X$ gates matters, since some qubits used as controls are also targets of other gates! You have to reverse the order in which the gates are applied, iterating through the pairs from right to left.

In [ ]:
def oracle_periodic_p(x: Qubits, y: Qubits, p: int) -> None:
    n = x.num_qubits
    for ind in range(n - p):
        x[ind].x(cond=x[ind + p])
    y.x(cond=~x[:n - p])
    for ind in range(n - p - 1, -1, -1):
        x[ind].x(cond=x[ind + p])

## Problem 7. Is bit string periodic?

We need to check whether for any value of $P$ the bit string is periodic with period $P$. 
We can use the solution to the previous problem as a building block to check periodicity for a specific $P$, but how do we build a complete solution out of these blocks?

You can express the function we're evaluating as "the bit string is periodic with period $1$" OR
"the bit string is periodic with period $2$" OR ... OR "the bit string is periodic with period $N − 1$".
Then, you have to allocate $N - 1$ auxiliary qubits and use them to store the evaluation results for the condition for each value of the period from $1$ to $N - 1$.

After this, you need to compute the OR of the states of these auxiliary qubits: if at least one of them is $1$, the bit string is periodic.
You can do this using the "Implement the OR oracle" exercise in the Oracles kata, by checking whether the values of all auxiliary qubits are $0$ and then negating the result.

Finally, you have to uncompute the changes you did to the auxiliary qubits to return them to the $\ket{0}$ state before releasing them.

In [ ]:
def oracle_periodic(x: Qubits, y: Qubits) -> None:
    n = x.num_qubits
    period_p = Qubits(n - 1, "period_p", x.qpu)
    for p in range(1, n):
        oracle_periodic_p(x, period_p[p - 1], p)
    y.x(cond=~period_p)
    y.x()
    for p in range(1, n):
        oracle_periodic_p(x, period_p[p - 1], p)
    period_p.release()

## Problem 8. Does bit string contain substring at position P?

In this problem, the value of the function we're evaluating does not depend on the state of the qubits before qubit $P$ or after qubit $P + K - 1$. This means that we can just ignore them and consider only the qubits that matter - `x[P:P + K - 1]`.

Once we do that, the problem becomes much simpler: flip the state of the target qubit if the input qubits are in the given state. That's the definition of a controlled gate with arbitrary control pattern. In Workbench, we can apply that by converting the bit pattern $pattern$ into a little-endian integer and using it as part of the condition. 

In [ ]:
def oracle_contains_substring_at_p(x: Qubits, y: Qubits, pattern: list[bool], p: int) -> None:
    K = len(pattern)
    pattern_int = sum([2 ** ind if pattern[ind] else 0 for ind in range(K)])
    y.x(cond=x[p:p + K] == pattern_int)

## Problem 9. Pattern matching

This problem is similar to the previous one: the function we're evaluating depends only on the states of a subset of qubits, so we can extract that subset and ignore the rest. The main difference is the way we're extracting the set of qubits we need to use as controls.

In Workbench, in addition to slicing a quantum register, you can get the list of qubits at given indices using `x[indices]`. We can use this to get the register to use as the control.

In [ ]:
def oracle_pattern_matching(x: Qubits, y: Qubits, indices: list[int], pattern: list[bool]) -> None:
    pattern_int = sum([2 ** ind if pattern[ind] else 0 for ind in range(len(pattern))])
    y.x(cond=x[indices] == pattern_int)

## Problem 10. Does bit string contain substring?

This problem is very similar to problem 7 in which we checked whether the string is periodic. 
Same as there, we have a building block that checks our condition (problem 8, in which we checked whether the bit string contains a given pattern at the specific position), and our function is an OR of conditions that apply with different parameters (the given pattern can start with position $0$, $1$, $2$, and so on).

The solution is similar as well:

1. Allocate $N - K + 1$ auxiliary qubits, one for each position that can be the beginning of the pattern.
2. Evaluate the condition with each possible position as the beginning of the pattern, and store the results in these qubits.
3. Evaluate the overall function as an OR of the values of the auxiliary qubits.
4. Uncompute the changes done to the states of the auxiliary qubits before releasing them.

In [ ]:
def oracle_contains_substring(x: Qubits, y: Qubits, pattern: list[bool]) -> None:
    n = len(x)
    k = len(pattern)
    substring_p = Qubits(n - k + 1, "substring_p", x.qpu)
    # Evaluate individual checks
    for p in range(n - k + 1):
        oracle_contains_substring_at_p(x, substring_p[p], pattern, p)
    # Compute OR of the results of individual checks
    y.x(cond=~substring_p)
    y.x()
    # Uncompute
    for p in range(n - k + 1):
        oracle_contains_substring_at_p(x, substring_p[p], pattern, p)
    substring_p.release()

## Problem 11. Is bit string balanced?

We will need a helper routine to count the number of $1$ bits in the bit string. This routine will act on a quantum register that stores a little-endian integer and increment this integer. We could implement this operation by hand, but Workbench offers several Qubricks that implement quantum addition, including adding a classical constant to a quantum register. We can use any of these, for example, `NaiveAdder`.

With this helper operation implemented, solving the task becomes clear:

1. Allocate the auxiliary qubits to store the counter - the number of $1$ bits in the bit string. The maximum number of $1$ bits is $N$, the length of the bit string itself. To store this number, we need a register of length `n.bit_length()`.
2. Count the number of $1$ bits in the input register. To do this, you need to increment the counter register for each qubit of the input register that is in $\ket{1}$ state - in other words, to add each qubit of the input register to the sum register.
3. Check whether the number of $1$ bits is exactly $\frac{N}{2}$, and flip the target qubit if it is using `cond=counter == n // 2` argument to the controlled gate.
4. As usual, uncompute the changes done to the auxiliary qubits before releasing them.

In [ ]:
def oracle_balanced(x: Qubits, y: Qubits) -> None:
    n = x.num_qubits
    sum = Qubits(n.bit_length(), "sum", x.qpu)
    for xi in x:
        sum += xi
    
    y.x(cond=sum == n // 2)

    for xi in x:
        sum -= xi
    sum.release()

## Problem 12. Majority function

This task will also rely on counting $1$ bits in the bit string, and we'll use the same approach to it as we did in the previous task. However, this time the condition we need to check is different: instead of checking whether the number of $1$ bits is a fixed number, we want to check whether it's greater than a constant $\frac{N - 1}2$. Workbench offers a built-in way to compare an integer stored in a qubit register to a constant, so you don't need to implement that comparison by hand!

In [ ]:
def oracle_majority(x: Qubits, y: Qubits) -> None:
    n = x.num_qubits
    sum = Qubits(n.bit_length(), "sum", x.qpu)
    for xi in x:
        sum += xi
    
    with sum > n // 2 as maj:
        y.x(cond=maj)

    for xi in x:
        sum -= xi        
    sum.release()

## Problem 13. Is bit sum divisible by 3?

Let's start by considering a simpler version of the problem. How would you check whether the number of bits equal to $1$ is divisible by $2$? 

You can just iterate over the input qubits and do a sequence of controlled $X$ gates with each of them as the control and the output qubit as the target. The controlled $X$ gate does a controlled flip of the state, which is the same as taking sum of the bits modulo $2$, so in the end you just flip it again to become $1$ if the sum modulo $2$ was $0$ and vice versa.

You can use the same approach here, but you need to implement addition modulo $3$ as an elementary operation.
To do this, you use a two-qubit register (in little-endian encoding, storing the least significant bit first and the most significant bit second) to store numbers $0$, $1$, $2$. Then, you figure out the rules of updating the register. The state of the register needs to change as follows with each increment:

$$\ket{0_{lsb}0_{msb}} \rightarrow \ket{1_{lsb}0_{msb}} \rightarrow \ket{0_{lsb}1_{msb}} \rightarrow \ket{0_{lsb}0_{msb}}$$

(The basis state $\ket{1_{lsb}1_{msb}}$ needs to remain unchanged to keep the transformation unitary, though we'll never use it in our solution.) One way to do this is as follows:

1. Start by updating the least significant bit `lsb`.  
   The `lsb` changes if incoming `msb` is $0$ (the first two transitions), and remains unchanged otherwise (the last transition). We can implement this using controlled-on-zero $X$ gate.
2. Then, update the `msb` bit.
   It changes if the updated `lsb` bit is $0$ (the last two transitions), which can be done using another controlled-on-zero $X$ gate.

With `IncrementMod3` operation implemented as a Qubrick, you iterate over the input qubits and count the number of $1$ bits among them modulo $3$. If the counter register ends up in $\ket{00}$ state, the number of $1$ bits is divisible by $3$, and you need to flip the target qubit.

In [ ]:
from psiqdk.workbench import Qubrick

class IncrementMod3(Qubrick):
    def _compute(self, reg: Qubits, cond: Qubits) -> None:
        '''Perform increment of a little-endian register modulo 3.'''
        lsb = reg[0]
        msb = reg[1]
        lsb.x(cond=~msb | cond)
        msb.x(cond=~lsb | cond)

def oracle_bit_sum_divisible_by_three(x: Qubits, y: Qubits) -> None:
    bit_sum = Qubits(2, "bit_sum", x.qpu)
    inc = IncrementMod3()
    for xi in x:
        inc.compute(bit_sum, cond=xi)
    y.x(cond=bit_sum == 0)
    for xi in x:
        inc.uncompute()
    bit_sum.release()

## Problem 14. Is the number divisible by 3?

This problem is similar to the previous one, but this time, the solution needs to be slightly more complicated: instead of just iterating through the bits and counting the number of $1$ bits modulo $3$, you need to calculate the remainder of dividing the number by $3$. You can use the fact that consecutive powers of $2$ give remainders $1$ and $-1$ in turn: 

$$2^0 \equiv 1 \mod 3, 2^1 = 2 \equiv -1 \mod 3, 2^2 = 4 \equiv 1 \mod 3, 2^3 = 8 \equiv -1 \mod 3, ...$$

You start iterating from the least significant bit of the input number, performing controlled increment of the counter register with qubits 0, 2, 4, ... as controls, and controlled decrement with qubits 1, 3, 5, ... as controls.

Finally, the number itself will be divisible by $3$ only if the counter qubits are in the $\ket{00}$ state.

In the code, we'll use the `IncrementMod3` Qubrick defined in the previous problem as a building block. We'll need to both increment and decrement numbers modulo $3$ this time. Fortunately, decrement is just an adjoint of increment, and Workbench can generate it for you automatically, without you implementing it by hand: you just need to call the `compute` method of the `IncrementMod3` with an additional argument `dagger=True`.

In [ ]:
def oracle_number_divisible_by_three(x: Qubits, y: Qubits) -> None:
    bit_sum = Qubits(2, "bit_sum", x.qpu)
    inc = IncrementMod3()
    for ind in range(x.num_qubits):
        if ind % 2 == 0:
            inc.compute(bit_sum, cond=x[ind])
        else:
            inc.compute(bit_sum, cond=x[ind], dagger=True)
    y.x(cond=bit_sum == 0)
    for ind in range(x.num_qubits):
        inc.uncompute()
    bit_sum.release()

> Copyright (c) 2026 PsiQuantum